# 03 Feature Engineering
In this notebook, we transform raw weather and crop features, encode categorical variables, compute weather-pest interaction features, and prepare training/testing sets.


In [1]:
import pandas as pd
import numpy as np
import time
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

start=time.time()
df = pd.read_csv('../datasets/pest_weather.csv')
print(df.shape)
print('Initial columns:', df.columns.tolist())
print(f"Time Taken: {time.time() - start:.2f} seconds")


(249700, 11)
Initial columns: ['State', 'District', 'Crop', 'Crop_Category', 'Sowing_Season', 'Season', 'Rainfall_mm', 'Temperature', 'Humidity', 'Pest_Name', 'Outbreak']
Time Taken: 1.13 seconds


In [3]:
# Weather Interaction Features
df['THI'] = 0.8 * df['Temperature'] + (df['Humidity'] / 100.0) * (df['Temperature'] - 14.3) + 46.4 # Temperature Humidity Index
df['Rainfall_Humidity_Ratio'] = df['Rainfall_mm'] / (df['Humidity'] + 1.0)

print('Engineered Features Sample:')
print(df[['Temperature', 'Humidity', 'Rainfall_mm', 'THI', 'Rainfall_Humidity_Ratio']].head())


df.head()
df.info()

Engineered Features Sample:
   Temperature  Humidity  Rainfall_mm        THI  Rainfall_Humidity_Ratio
0        32.44     71.81       599.71  85.378334                 8.236643
1        23.24     54.74        37.13  69.885756                 0.666128
2        28.01     57.19        14.75  76.648749                 0.253480
3        28.23     62.01         9.80  77.621993                 0.155531
4        20.74     65.62         3.86  67.217928                 0.057941
<class 'pandas.DataFrame'>
RangeIndex: 249700 entries, 0 to 249699
Data columns (total 13 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   State                    249700 non-null  str    
 1   District                 249700 non-null  str    
 2   Crop                     249700 non-null  str    
 3   Crop_Category            249700 non-null  str    
 4   Sowing_Season            249700 non-null  str    
 5   Season                   249700 non-nu

In [4]:
from sklearn.preprocessing import LabelEncoder

df_encoded = df.copy()

# Encode Target
target_encoder = LabelEncoder()
df_encoded["Outbreak"] = target_encoder.fit_transform(df_encoded["Outbreak"])

# Encode Categorical Features
categorical_cols = [
    "State",
    "District",
    "Crop",
    "Crop_Category",
    "Sowing_Season",
    "Season",
    "Pest_Name"
]

encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    encoders[col] = le

print("Encoding Completed!")
df_encoded.head()

Encoding Completed!


,State,District,Crop,Crop_Category,Sowing_Season,Season,Rainfall_mm,Temperature,Humidity,Pest_Name,Outbreak,THI,Rainfall_Humidity_Ratio
0,0,9,3,0,1,0,599.71,32.44,71.81,20,0,85.378334,8.236643
1,0,9,3,0,1,2,37.13,23.24,54.74,6,2,69.885756,0.666128
2,0,9,3,0,1,0,14.75,28.01,57.19,18,0,76.648749,0.253480
3,0,9,3,0,1,2,9.80,28.23,62.01,20,0,77.621993,0.155531
4,0,9,3,0,1,2,3.86,20.74,65.62,6,1,67.217928,0.057941


In [5]:
df_encoded.info()

<class 'pandas.DataFrame'>
RangeIndex: 249700 entries, 0 to 249699
Data columns (total 13 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   State                    249700 non-null  int64  
 1   District                 249700 non-null  int64  
 2   Crop                     249700 non-null  int64  
 3   Crop_Category            249700 non-null  int64  
 4   Sowing_Season            249700 non-null  int64  
 5   Season                   249700 non-null  int64  
 6   Rainfall_mm              249700 non-null  float64
 7   Temperature              249700 non-null  float64
 8   Humidity                 249700 non-null  float64
 9   Pest_Name                249700 non-null  int64  
 10  Outbreak                 249700 non-null  int64  
 11  THI                      249700 non-null  float64
 12  Rainfall_Humidity_Ratio  249700 non-null  float64
dtypes: float64(5), int64(8)
memory usage: 24.8 MB


In [ ]:
# Categorical Encoding
le_outbreak = LabelEncoder()
df['Outbreak_Target'] = le_outbreak.fit_transform(df['Outbreak']) # Low=0, Medium=1, High=2 (or similar)

# Encode categorical predictors
cat_cols = ['State', 'District', 'Crop', 'Crop_Category', 'Sowing_Season', 'Season', 'Pest_Name']
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

print('Encoded dataset shape:', df_encoded.shape)


In [ ]:
# Save processed features
# df_encoded.to_csv('../datasets/processed_pest_weather.csv', index=False)
print('Feature engineering complete.')


In [6]:
from sklearn.model_selection import train_test_split

# Features and Target
X = df_encoded.drop("Outbreak", axis=1)
y = df_encoded["Outbreak"]

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (199760, 12)
X_test : (49940, 12)
y_train: (199760,)
y_test : (49940,)


In [7]:
X_train.to_csv("../datasets/X_train.csv", index=False)
X_test.to_csv("../datasets/X_test.csv", index=False)

import pandas as pd

pd.DataFrame(y_train, columns=["Outbreak"]).to_csv("../datasets/y_train.csv", index=False)
pd.DataFrame(y_test, columns=["Outbreak"]).to_csv("../datasets/y_test.csv", index=False)

print("Datasets Saved Successfully!")

Datasets Saved Successfully!
